# Baseline-модели рекомендаций туров

Сравниваем четыре подхода: quality baseline (туры того же города по рейтингу звёзд), content-based по структурным признакам (city, category, stars, meal, price), TF-IDF по текстовым описаниям и их комбинация.

**Данные:** `data/tours.json`, только активные туры.

---
## 1. Загрузка и подготовка данных

In [14]:
import sys
import json
import pickle
import numpy as np
from pathlib import Path

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

sys.path.insert(0, str(Path("..").resolve()))
from src.models.features import build_feature_matrices
from src.models.metrics import (
    category_hit_rate, catalog_coverage, mean_similarity,
    intra_list_diversity, personalization, popularity_metrics,
)
from src.models.recommender import get_similar_tours, get_popularity_recs

DATA_PATH = Path("../data/tours.json")
ARTIFACTS = Path("../artifacts/exp02")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

In [15]:
with open(DATA_PATH, encoding="utf-8") as f:
    df_all = pd.DataFrame(json.load(f))

df = df_all[df_all["status"] == "active"].reset_index(drop=True)
print(f"Активных туров: {len(df)}")
df[["id", "city_id", "category", "hotel_stars", "meal_type", "price", "duration_nights"]].head()

Активных туров: 170


,id,city_id,category,hotel_stars,meal_type,price,duration_nights
0,362b3800-ccd0-47d2-b8cc-158bd0d22742,istanbul,comfort,4,all_inclusive,1049.87,8
1,85557a31-8f84-459f-89cf-5ddccdef0590,new_york,comfort,4,none,2744.72,11
2,2f1de70e-2b56-4e38-80d2-af9aaf2edb17,bali,budget,3,all_inclusive,1564.80,14
3,7aed5208-d601-41e1-bdbf-fbd10552f0ee,istanbul,budget,2,breakfast,498.27,10
4,248ee8e7-1e6a-480c-9c21-817ef277a724,dubai,luxury,5,breakfast,7174.88,8


---
## 2. Инженерия признаков

In [16]:
struct_matrix, tfidf_matrix, scaler, tfidf = build_feature_matrices(df)
print(f"struct_matrix: {struct_matrix.shape}")
print(f"tfidf_matrix:  {tfidf_matrix.shape}")

struct_matrix: (170, 19)
tfidf_matrix:  (170, 200)


In [17]:
cat_cols = pd.get_dummies(df[["city_id", "category", "meal_type"]], dtype=float).columns.tolist()
num_cols = ["hotel_stars", "duration_nights", "log_price"]

print(f"struct_matrix — {len(cat_cols) + len(num_cols)} признаков\n")
print(f"One-hot ({len(cat_cols)}):")
for c in cat_cols:
    print(f"  {c}")
print(f"\nЧисловые, MinMaxScaler ({len(num_cols)}):")
for c in num_cols:
    note = "  ← np.log1p(price)" if c == "log_price" else ""
    print(f"  {c}{note}")

struct_matrix — 19 признаков

One-hot (16):
  city_id_bali
  city_id_barcelona
  city_id_cairo
  city_id_dubai
  city_id_istanbul
  city_id_maldives
  city_id_new_york
  city_id_paris
  city_id_prague
  city_id_tokyo
  category_budget
  category_comfort
  category_luxury
  meal_type_all_inclusive
  meal_type_breakfast
  meal_type_none

Числовые, MinMaxScaler (3):
  hotel_stars
  duration_nights
  log_price  ← np.log1p(price)


**Состав признаков:** 16 one-hot (10 городов + 3 категории + 3 типа питания) + 3 числовых = 19. Цена входит как `log_price = np.log1p(price)` — устраняет правый скос (skewness 2.x→0.x, см. exp01 секция 4). После этого все три числовых признака нормируются MinMaxScaler в [0, 1].

In [18]:
STRUCT_WEIGHT = 0.7
TEXT_WEIGHT   = 0.3

---
## 3. Матрицы схожести

In [19]:
sim_struct   = cosine_similarity(struct_matrix)
sim_tfidf    = cosine_similarity(tfidf_matrix)
sim_combined = STRUCT_WEIGHT * sim_struct + TEXT_WEIGHT * sim_tfidf

print(f"Матрицы схожести построены: {sim_struct.shape}")

Матрицы схожести построены: (170, 170)


---
## 4. Функции рекомендаций

In [20]:
def quality_baseline(tour_id: str, top_k: int = 5) -> pd.DataFrame:
    """Возвращает top_k туров того же города, отсортированных по hotel_stars DESC."""
    idx = df.index[df["id"] == tour_id][0]
    cands = get_popularity_recs(df, idx, top_k)
    return df.iloc[cands][["name", "city_name", "category", "hotel_stars", "meal_type", "price"]].reset_index(drop=True)

---
## 5. Примеры рекомендаций

In [21]:
query_tour = df[df["city_id"] == "barcelona"].iloc[0]
print(f"Тур:      {query_tour['name']}")
print(f"Город:    {query_tour['city_name']}")
print(f"Категория:{query_tour['category']}")
print(f"Отель:    {query_tour['hotel_name']} ({query_tour['hotel_stars']} stars)")
print(f"Питание:  {query_tour['meal_type']}")
print(f"Цена:     €{query_tour['price']:.0f}")
print(f"Описание: {query_tour['description'][:120]}...")

Тур:      Classic Barcelona — 6 nights
Город:    Barcelona
Категория:comfort
Отель:    Hotel Eixample (4 stars)
Питание:  all_inclusive
Цена:     €1762
Описание: Hotel Eixample погружает вас в самый модный район Барселоны — туда, где архитектура Гауди соседствует с бутиками и ресто...


In [22]:
qid = query_tour["id"]

print("\nQuality baseline (туры того же города по рейтингу звёзд)")
display(quality_baseline(qid))

print("\nContent-based: структурные признаки")
display(get_similar_tours(df, qid, sim_struct))

print("\nContent-based: TF-IDF по описанию")
display(get_similar_tours(df, qid, sim_tfidf))

print("\nContent-based: структура + TF-IDF")
display(get_similar_tours(df, qid, sim_combined))


Quality baseline (туры того же города по рейтингу звёзд)


,name,city_name,category,hotel_stars,meal_type,price
0,Elite Barcelona — 6 nights,Barcelona,luxury,5,breakfast,2419.24
1,Exclusive Barcelona — 5 nights,Barcelona,luxury,5,none,2307.30
2,Exclusive Barcelona — 10 nights,Barcelona,luxury,5,all_inclusive,1984.50
3,Premium Barcelona — 9 nights,Barcelona,luxury,5,all_inclusive,2640.54
4,Luxury Barcelona — 6 nights,Barcelona,luxury,5,none,2451.20



Content-based: структурные признаки


,name,city_name,category,hotel_stars,meal_type,price,similarity
0,Standard Barcelona — 5 nights,Barcelona,comfort,4,all_inclusive,1915.36,0.999
1,Standard Barcelona — 9 nights,Barcelona,comfort,4,all_inclusive,1757.37,0.990
2,Premium Barcelona — 9 nights,Barcelona,luxury,5,all_inclusive,2640.54,0.714
3,Classic New York — 8 nights,New York,comfort,4,all_inclusive,3376.01,0.713
4,Popular New York — 8 nights,New York,comfort,4,all_inclusive,3252.32,0.713



Content-based: TF-IDF по описанию


,name,city_name,category,hotel_stars,meal_type,price,similarity
0,Exclusive Paris — 6 nights,Paris,luxury,5,all_inclusive,3931.00,0.361
1,Elite Maldives — 11 nights,Maldives,luxury,5,all_inclusive,13923.31,0.341
2,Premium Paris — 6 nights,Paris,luxury,5,none,2768.38,0.328
3,Luxury Barcelona — 6 nights,Barcelona,luxury,5,none,2451.20,0.322
4,Classic Barcelona — 8 nights,Barcelona,comfort,4,none,1328.15,0.314



Content-based: структура + TF-IDF


,name,city_name,category,hotel_stars,meal_type,price,similarity
0,Standard Barcelona — 5 nights,Barcelona,comfort,4,all_inclusive,1915.36,0.746
1,Standard Barcelona — 9 nights,Barcelona,comfort,4,all_inclusive,1757.37,0.736
2,Classic Barcelona — 8 nights,Barcelona,comfort,4,none,1328.15,0.588
3,Standard Barcelona — 8 nights,Barcelona,comfort,4,none,1163.71,0.586
4,Classic Paris — 6 nights,Paris,comfort,4,all_inclusive,1813.21,0.583


**Наблюдения (Barcelona comfort, 4-star, all_inclusive, €1762):**

- **Quality baseline:** все 5 рекомендаций — luxury 5-star. Ранжирует по звёздности, категорию запроса полностью игнорирует.
- **TF-IDF:** 4 из 5 — luxury (Paris, Maldives, Paris, Barcelona); similarity 0.31–0.36 — текст не кодирует ценовой сегмент.
- **Структурная:** топ-2 идеальны (Barcelona comfort 4-star all_inclusive, sim≈1.0). Позиция 3 — Premium Barcelona luxury 5-star (другая категория, sim=0.714). Позиции 4–5 — New York comfort 4-star с одинаковым score 0.713 — ничья, порядок случаен.
- **Combined:** топ-2 совпадают со структурной. На позиции 3 появляется Classic Barcelona comfort 4-star (none) вместо нью-йоркских туров — TF-IDF отдал предпочтение описаниям Барселоны перед структурно-идентичными, но географически чужими турами. Позиция 5 — Classic Paris comfort 4-star: TF-IDF добавляет географическое разнообразие.

### Дополнительные примеры: luxury и budget запросы

Сравниваем структурную модель и combined на двух полярных профилях.

In [23]:
extra_queries = [
    df[df["city_id"] == "maldives"].iloc[0],  # luxury
    df[df["city_id"] == "prague"].iloc[0],    # budget
]

for q in extra_queries:
    print("=" * 65)
    print(f"Запрос: {q['name']}")
    print(f"        {q['category']} | {q['hotel_stars']} stars | {q['meal_type']} | €{q['price']:.0f}")
    print()
    qid = q["id"]
    print("Структурные признаки:")
    display(get_similar_tours(df, qid, sim_struct))
    print("Структура + TF-IDF:")
    display(get_similar_tours(df, qid, sim_combined))
    print()

Запрос: Elite Maldives — 11 nights
        luxury | 5 stars | all_inclusive | €13923

Структурные признаки:


,name,city_name,category,hotel_stars,meal_type,price,similarity
0,Premium Maldives — 11 nights,Maldives,luxury,5,all_inclusive,10740.15,1.000
1,Premium Maldives — 14 nights,Maldives,luxury,5,all_inclusive,9517.15,0.992
2,Elite Maldives — 13 nights,Maldives,luxury,5,all_inclusive,6309.15,0.992
3,Exclusive Dubai — 12 nights,Dubai,luxury,7,all_inclusive,13561.29,0.796
4,Exclusive Maldives — 14 nights,Maldives,luxury,5,breakfast,11907.49,0.796


Структура + TF-IDF:


,name,city_name,category,hotel_stars,meal_type,price,similarity
0,Premium Maldives — 11 nights,Maldives,luxury,5,all_inclusive,10740.15,0.825
1,Elite Maldives — 13 nights,Maldives,luxury,5,all_inclusive,6309.15,0.804
2,Premium Maldives — 14 nights,Maldives,luxury,5,all_inclusive,9517.15,0.767
3,Premium New York — 11 nights,New York,luxury,5,all_inclusive,7393.69,0.653
4,Exclusive Maldives — 14 nights,Maldives,luxury,5,breakfast,11907.49,0.651



Запрос: Budget Prague — 3 nights
        budget | 3 stars | breakfast | €690

Структурные признаки:


,name,city_name,category,hotel_stars,meal_type,price,similarity
0,Comfort Prague — 4 nights,Prague,comfort,4,breakfast,653.85,0.673
1,Economy Prague — 5 nights,Prague,budget,3,all_inclusive,973.52,0.672
2,Classic Prague — 5 nights,Prague,comfort,4,breakfast,598.08,0.670
3,Express Paris — 6 nights,Paris,budget,3,breakfast,1189.54,0.667
4,Budget Prague — 6 nights,Prague,budget,3,all_inclusive,783.05,0.667


Структура + TF-IDF:


,name,city_name,category,hotel_stars,meal_type,price,similarity
0,Classic Prague — 5 nights,Prague,comfort,4,breakfast,598.08,0.589
1,Economy Prague — 5 nights,Prague,budget,3,all_inclusive,973.52,0.578
2,Essential Cairo — 8 nights,Cairo,budget,3,breakfast,641.35,0.560
3,Economy Prague — 7 nights,Prague,budget,3,all_inclusive,867.08,0.545
4,Budget Prague — 6 nights,Prague,budget,3,all_inclusive,783.05,0.544


**Наблюдения (Maldives luxury, all_inclusive, €13923):**

- **Структурная:** sim=1.000 с первым двойником (Premium Maldives 11 nights, те же признаки). Позиции 4–5 — ничья: Exclusive Dubai 7-star all_inclusive и Exclusive Maldives breakfast оба с sim=0.796.
- **Combined:** переставляет туры Мальдив по тексту описания (Elite 13 nights выходит на позицию 2 вместо Premium 14 nights), на позиции 4 появляется New York luxury 5-star all_inclusive вместо Dubai 7-star — текстово ближе к maldives-стилю, хотя структурно одинаково далеко.

**Наблюдения (Prague budget, 3-star, breakfast, €690):**

- **Структурная:** топ-1 — Comfort Prague 4-star breakfast (категория comfort, не budget). При низком ценовом уровне структурные признаки плохо разграничивают budget и comfort — они слишком похожи по city/stars/meal. Это ограничение структурной модели на граничных профилях.
- **Combined:** на позиции 1 — Classic Prague comfort 4-star breakfast (другой тур, та же проблема категории). TF-IDF не исправляет категорию, но на позиции 3 появляется Essential Cairo budget 3-star breakfast — содержательно схожий профиль, которого структурная модель не видела в топ-5.

---
## 6. Оценка качества моделей

Оцениваем пять аспектов качества рекомендаций:

- **Category Hit Rate@5** — доля рекомендаций той же категории (luxury / comfort / budget), что и запрос. Отражает способность модели держать ценовой сегмент.
- **Coverage** — доля каталога, которая хотя бы раз попала в рекомендации. Выше — меньше «мёртвых зон».
- **Mean Similarity@5** — средняя cosine similarity топ-5 рекомендаций к запросу.
- **Intra-list Diversity@5 (ILD)** — среднее попарное расстояние (1 − similarity) между рекомендациями внутри одного топ-5. Выше — рекомендации разнообразнее; низкое значение сигнализирует о жёстких кластерах.
- **Personalization@5** — насколько разные списки получают разные запросы (1 − среднее пересечение). На этом датасете ожидаемо высокая у всех моделей из-за малого размера каталога — используем как проверочную метрику, не как дифференцирующую.

In [24]:
TOP_K = 5
pop_chr, pop_cov, pop_sim, pop_ild, pop_pers = popularity_metrics(df, sim_struct, TOP_K)

results = pd.DataFrame([
    {
        "Модель": "Quality baseline",
        "CHR@5": pop_chr,
        "Coverage": pop_cov,
        "Mean Sim@5": pop_sim,
        "ILD@5": pop_ild,
        "Personalization@5": pop_pers,
    },
    {
        "Модель": "Структурные признаки",
        "CHR@5": category_hit_rate(df, sim_struct, TOP_K),
        "Coverage": catalog_coverage(sim_struct, TOP_K),
        "Mean Sim@5": mean_similarity(sim_struct, TOP_K),
        "ILD@5": intra_list_diversity(sim_struct, TOP_K),
        "Personalization@5": personalization(sim_struct, TOP_K),
    },
    {
        "Модель": "TF-IDF (описание)",
        "CHR@5": category_hit_rate(df, sim_tfidf, TOP_K),
        "Coverage": catalog_coverage(sim_tfidf, TOP_K),
        "Mean Sim@5": mean_similarity(sim_tfidf, TOP_K),
        "ILD@5": intra_list_diversity(sim_tfidf, TOP_K),
        "Personalization@5": personalization(sim_tfidf, TOP_K),
    },
    {
        "Модель": "Структура + TF-IDF",
        "CHR@5": category_hit_rate(df, sim_combined, TOP_K),
        "Coverage": catalog_coverage(sim_combined, TOP_K),
        "Mean Sim@5": mean_similarity(sim_combined, TOP_K),
        "ILD@5": intra_list_diversity(sim_combined, TOP_K),
        "Personalization@5": personalization(sim_combined, TOP_K),
    },
]).set_index("Модель").round(3)

display(results)
results.to_csv(ARTIFACTS / "metrics_comparison.csv")

,CHR@5,Coverage,Mean Sim@5,ILD@5,Personalization@5
Модель,,,,,
Quality baseline,0.531,0.388,0.683,0.212,0.921
Структурные признаки,0.927,0.982,0.868,0.190,0.969
TF-IDF (описание),0.725,1.000,0.427,0.707,0.970
Структура + TF-IDF,0.933,1.000,0.713,0.368,0.972


### Анализ структурных кластеров

Проверяем утверждение о жёстких кластерах: считаем туры с `sim_struct ≥ 0.999` и смотрим, как TF-IDF оценивает их схожесть.

In [25]:
THRESHOLD = 0.999
n = len(sim_struct)
mask = (sim_struct >= THRESHOLD) & ~np.eye(n, dtype=bool)

n_in_cluster = int(mask.any(axis=1).sum())
n_pairs      = int(mask.sum() / 2)

twin_pairs   = np.argwhere(mask)
tfidf_within = float(np.mean([sim_tfidf[i, j] for i, j in twin_pairs])) if len(twin_pairs) else 0.0
tfidf_global = float((sim_tfidf.sum() - n) / (n * (n - 1)))

print(f"Туров со структурным двойником (sim≥{THRESHOLD}): {n_in_cluster} из {n} ({n_in_cluster/n*100:.0f}%)")
print(f"Уникальных пар с sim_struct≥{THRESHOLD}:          {n_pairs}")
print()
print(f"sim_tfidf внутри структурных кластеров: {tfidf_within:.3f}")
print(f"sim_tfidf по каталогу в среднем:        {tfidf_global:.3f}")
print(f"Разница:                                {tfidf_within - tfidf_global:+.3f}")

Туров со структурным двойником (sim≥0.999): 65 из 170 (38%)
Уникальных пар с sim_struct≥0.999:          44

sim_tfidf внутри структурных кластеров: 0.420
sim_tfidf по каталогу в среднем:        0.118
Разница:                                +0.302


**Наблюдение по кластерам:**

38% туров (65 из 170) имеют структурного двойника (sim_struct ≥ 0.999) — 44 уникальных пары. Внутри этих пар `sim_tfidf = 0.420` против среднего по каталогу `0.118`: структурные двойники похожи и по тексту описания (два luxury-тура на Мальдивах описаны схожим языком). TF-IDF не различает их — он **подтверждает** схожесть. Рост ILD у combined (0.190 → 0.368) объясняется иначе: TF-IDF тянет в топ-5 туры из других городов с похожими описаниями, добавляя географическое разнообразие, а не разбивая внутрикластерные ничьи.

---
## 7. Сохранение артефактов

In [26]:
np.save(ARTIFACTS / "sim_combined.npy", sim_combined)

with open(ARTIFACTS / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
with open(ARTIFACTS / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

df.to_json(ARTIFACTS / "tours_active.json", orient="records", force_ascii=False)

print("Сохранено:", [p.name for p in sorted(ARTIFACTS.iterdir())])

Сохранено: ['metrics_comparison.csv', 'scaler.pkl', 'sim_combined.npy', 'tfidf_vectorizer.pkl', 'tours_active.json']


---
## 8. Выводы и выбор финальной модели

Quality baseline слабее всего: CHR=0.531, Coverage=0.388, Personalization=0.921. При трёх категориях случайный бейзлайн дал бы CHR≈0.33, так что прирост есть, но более 60% каталога так и не появляется в рекомендациях. Низкий Personalization (0.921 против ~0.97 у content-based) объясняется тем, что все запросы из одного города получают идентичный список.

Структурные признаки резко улучшают картину: CHR=0.927, Coverage=0.982. Формализованные поля city/category/stars/meal/log_price несут почти всю информацию о ценовом сегменте. ILD=0.190 — самое низкое значение среди content-based моделей. Анализ кластеров подтверждает причину: 38% туров имеют структурного двойника (sim≥0.999), и внутри таких пар порядок выдачи случаен.

TF-IDF покрывает каталог полностью (Coverage=1.0), но CHR=0.725 — описания не кодируют ценовой сегмент так жёстко, как структурные поля. ILD=0.707 — высокое разнообразие внутри списка. Примечательно: sim_tfidf внутри структурных кластеров (0.420) заметно выше среднего по каталогу (0.118) — структурные двойники описаны похоже, TF-IDF подтверждает их схожесть, а не различает.

Комбинированная модель (0.7 × sim_struct + 0.3 × sim_tfidf): CHR=0.933, Coverage=1.000, ILD=0.368. Coverage стал идеальным (1.000), ILD вырос с 0.190 до 0.368 (+94%). Рост ILD объясняется не разбиванием внутрикластерных ничьих, а тем, что TF-IDF тянет в топ-5 туры из других городов с похожими описаниями — это видно в примерах (Barcelona comfort → Paris появляется на позиции 5).

**Финальная модель для сервиса:** `combined`. По CHR не хуже структурной (0.933 vs 0.927), Coverage идеальная, ILD вдвое выше. Веса 0.7/0.3 — стартовая точка; grid search в `exp03_weight_tuning.ipynb` проверит оптимальную конфигурацию.

**Направления улучшений:**
- **Sentence-embeddings вместо TF-IDF:** TF-IDF работает на совпадении токенов и не улавливает семантику ("экзотический курорт" ≠ "пляжный отдых" для модели). Mean Similarity у TF-IDF = 0.427 против 0.868 у структурной — текстовый сигнал слабый. Sentence-embeddings должны дать более осмысленное текстовое сходство.
- **Жёсткая фильтрация по категории перед ранжированием:** Prague budget пример показал, что структурная модель ставит comfort 4-star на позицию 1 для budget-запроса — признаки не разграничивают эти категории на низком ценовом уровне. Фильтр `category == query.category` до ранжирования устранит эту проблему явно.